# Exploration des DECP consolidees

Premiere phase d'exploration du jeu principal du projet : les donnees essentielles de la commande
publique, version consolidee et publiee en Parquet sur data.gouv.fr.

**Objectif** : ne plus reprendre les chiffres des rapports officiels, mais les mesurer nous-memes,
et comprendre la structure du fichier avant d'ecrire la moindre ligne de pipeline.

**Prerequis** : `make donnees-decp` a ete lance, le fichier est dans `donnees/brut/decp.parquet`.

**Pourquoi un notebook** : cette etape est de l'exploration. On pose une question, on regarde, la
reponse change la question suivante. Une fois les mesures stabilisees, elles sont recopiees dans
`mesures/decp_qualite.py`, qui les rejoue de facon reproductible et datee.

## Pourquoi ces outils, et pas d'autres

Le choix se fait a une echelle precise : **3,28 millions de lignes, 66 colonnes, 236 Mo en Parquet,
sur un portable de 16 Go**. A une autre echelle, la reponse changerait, et c'est ce qu'il faut
savoir dire.

| Option | Avantages | Limites ici | Verdict |
|---|---|---|---|
| **DuckDB sur Parquet** | aucun serveur, lit le fichier sur place, ne charge que les colonnes utiles, SQL complet, agregations en dizaines de millisecondes | mono-machine, ne sert pas des utilisateurs simultanes | **retenu pour l'exploration** |
| pandas | ecosysteme immense, indispensable pour manipuler des resultats | charge tout en memoire avant d'agreger : plusieurs Go de RAM et plusieurs secondes pour un simple group by | retenu, mais pour **afficher** les resultats, pas pour calculer |
| polars | tres rapide, lit aussi le Parquet, API moderne et typee | une API de plus a apprendre, alors que le SQL est deja exige par le memoire et servira tel quel dans dbt | ecarte a ce stade |
| PySpark | passe a l'echelle sur un cluster, standard en entreprise | concu pour ce qui ne tient pas sur une machine ; ici tout tient en RAM. Demarrer une JVM et un cluster couterait plus que le calcul lui-meme | ecarte, et c'est un argument de cout du memoire |
| PostgreSQL | la base de l'application finale, transactionnelle | il faudrait installer, creer un schema et importer 3,3 M de lignes avant la premiere question | retenu plus tard, pour **servir** le site |

**A quelle echelle la reponse changerait** : au-dela d'une centaine de Go, ou si plusieurs machines
devaient traiter le meme jeu en parallele, Spark ou un entrepot distribue redeviendraient pertinents.
En dessous, ils ajoutent du cout et de la complexite sans gain mesurable. Ce seuil sera mesure et
cite dans le memoire, plutot qu'affirme.

**Pourquoi pas un jeu tout prepare de Kaggle ou de Hugging Face** : il en existe sur les marches
publics, mais ils sont figes, souvent deja nettoyes par quelqu'un d'autre, et sans tracabilite vers
la source officielle. Le sujet du memoire porte justement sur la chaine complete, du fichier public
brut jusqu'au service. Partir d'un jeu pre-nettoye supprimerait le probleme a demontrer. Hugging
Face servira en revanche en phase 6, pour les **modeles** pre-entraines.

## 1. Se connecter aux donnees

DuckDB lit le fichier Parquet **sur place**. Aucune base n'est demarree, aucune donnee n'est
importee ni copiee. C'est le premier argument de cout du memoire : la meme exploration sur une base
serveur demanderait d'installer PostgreSQL, de creer un schema et d'importer 3,3 millions de lignes
avant de pouvoir poser la premiere question.

In [1]:
import time
from pathlib import Path

import duckdb

RACINE = Path.cwd().parent if Path.cwd().name == "analyses" else Path.cwd()
FICHIER = RACINE / "donnees" / "brut" / "decp.parquet"

con = duckdb.connect()
# On enregistre le fichier comme une table nommee, plutot que de coller son chemin dans chaque
# requete SQL. Deux avantages : les requetes restent lisibles, et aucune chaine n'est interpolee
# dans du SQL (pas d'injection possible si le chemin venait un jour de l'exterieur).
con.register("decp", con.read_parquet(str(FICHIER)))

print(f"fichier   : {FICHIER.name}")
print(f"taille    : {FICHIER.stat().st_size / 1e6:.1f} Mo")
print(f"duckdb    : {duckdb.__version__}")

fichier   : decp.parquet
taille    : 247.6 Mo
duckdb    : 1.5.5


## 2. Le volume, et une mesure qui semble truquee

Le comptage des lignes est quasi instantane. Ce n'est pas de la magie et il faut savoir l'expliquer :
Parquet range le nombre de lignes dans ses metadonnees, donc DuckDB repond sans lire une seule
donnee. Une agregation qui lit vraiment les colonnes prend plusieurs dizaines de millisecondes, et
une comparaison ligne a ligne sur les 66 colonnes prend plusieurs secondes.

In [2]:
debut = time.perf_counter()
lignes = con.sql("select count(*) from decp").fetchone()[0]
duree_comptage = time.perf_counter() - debut

print(f"{lignes:>12,} lignes".replace(",", " "))
print(f"comptage en {duree_comptage:.4f} s (metadonnees Parquet, aucune donnee lue)")

con.sql("""
    select
        count(*)                     as lignes,
        count(distinct uid)          as marches_distincts,
        count(distinct acheteur_id)  as acheteurs,
        count(distinct titulaire_id) as titulaires
    from decp
""").df()

   3 283 035 lignes
comptage en 0.0094 s (metadonnees Parquet, aucune donnee lue)


,lignes,marches_distincts,acheteurs,titulaires
0,3283035,1845800,29069,216335


### Comparaison avec les chiffres du producteur

Le producteur publie ses propres statistiques dans `statistiques-marches.json`. On les confronte aux
notres. Les ecarts ne sont pas des erreurs : ils viennent de definitions differentes, et c'est
exactement ce qu'il faut savoir expliquer devant un jury.

| Indicateur | Producteur | Mesure ici |
|---|---|---|
| Lignes | 3 283 035 | identique |
| Marches | 1 833 468 | identique, une fois filtre sur l'etat actuel |
| Colonnes | 68 | **66 dans le fichier** |
| Acheteurs uniques | 29 042 | 29 069 |
| Titulaires uniques | 217 199 | 216 335 |

A verifier plus tard : d'ou viennent les deux colonnes manquantes, et pourquoi les comptages
d'acheteurs et de titulaires different de quelques dizaines.

## 3. Le piege des doublons

3,28 millions de lignes pour 1,85 million d'identifiants. La conclusion facile serait d'annoncer
44 % de doublons. Elle serait fausse, et c'est une erreur classique sur ce jeu de donnees.

Trois causes possibles a plusieurs lignes portant le meme identifiant :

1. l'**historique des modifications** : chaque avenant ajoute une ligne, avec son `modification_id` ;
2. les **groupements d'entreprises** : un marche attribue a trois entreprises donne trois lignes ;
3. les **vrais doublons** : la meme information publiee deux fois.

Seul le troisieme cas est un defaut de qualite. On les separe.

In [3]:
# Combien de lignes porte un identifiant donne ?
con.sql("""
    with par_uid as (select uid, count(*) as lignes from decp group by 1)
    select lignes as lignes_pour_un_uid, count(*) as nb_uid
    from par_uid group by 1 order by 1 limit 8
""").df()

,lignes_pour_un_uid,nb_uid
0,1,1420063
1,2,200493
2,3,84950
3,4,47643
4,5,22706
5,6,19656
6,7,8629
7,8,8937


In [4]:
# On descend d'un cran : pour un meme couple (marche, modification), plusieurs lignes signifient
# soit plusieurs titulaires (normal), soit un vrai doublon (defaut).
con.sql("""
    with par_couple as (
        select
            uid,
            modification_id,
            count(*)                     as lignes,
            count(distinct titulaire_id) as titulaires
        from decp
        group by 1, 2
    )
    select
        sum(case when lignes > 1 and lignes = titulaires then 1 else 0 end) as multi_titulaires,
        sum(case when lignes > 1 and lignes > titulaires then 1 else 0 end) as doublons_residuels
    from par_couple
""").df()

,multi_titulaires,doublons_residuels
0,273494.0,7764.0


In [5]:
# Verification de notre lecture : en ne gardant que l'etat actuel de chaque marche, retrouve-t-on
# le nombre officiel de 1 833 468 marches ? Si oui, notre comprehension du fichier est validee par
# un chiffre que nous n'avons pas choisi.
con.sql("""
    select count(*) as lignes, count(distinct uid) as marches
    from decp where donneesActuelles
""").df()

,lignes,marches
0,2114675,1833468


**Resultat** : 1 833 468 marches, exactement le chiffre du producteur. Les lignes en trop sont donc
de l'historique et des groupements, pas du bruit. Les vrais doublons residuels sont environ 7 800,
et aucune ligne n'est strictement identique a une autre sur les 66 colonnes.

Consequence pour le pipeline : la table de travail ne sera pas une deduplication brutale, mais un
filtre sur l'etat actuel, avec l'historique conserve a part.

## 4. Les montants

C'est la colonne la plus utilisee du jeu, et la plus abimee.

In [6]:
con.sql("""
    select
        sum(case when montant is null then 1 else 0 end) as absents,
        sum(case when montant <= 0 then 1 else 0 end)    as negatifs_ou_nuls,
        sum(case when montant > 1e9 then 1 else 0 end)   as superieurs_au_milliard,
        round(min(montant), 2)                           as minimum,
        round(median(montant), 2)                        as mediane,
        round(max(montant), 2)                           as maximum
    from decp
""").df()

,absents,negatifs_ou_nuls,superieurs_au_milliard,minimum,mediane,maximum
0,48596.0,59588.0,2197.0,-2676107.0,167832.96,1.000000e+11


In [7]:
# Le producteur signale lui-meme les montants douteux. Utiliser son travail plutot que de le
# refaire, puis comparer notre propre detection a la sienne en phase 5.
con.sql("""
    select coalesce(montant_anomalie, '(aucune)') as anomalie, count(*) as lignes
    from decp group by 1 order by lignes desc
""").df()

,anomalie,lignes
0,(aucune),3151807
1,suspect,98815
2,aberrant,32413


In [8]:
# A quoi ressemblent concretement les montants aberrants ?
con.sql("""
    select objet, montant, montant_anomalie_raisons, acheteur_nom
    from decp
    where montant > 1e9
    order by montant desc
    limit 5
""").df()

,objet,montant,montant_anomalie_raisons,acheteur_nom
0,Accord cadre à bons de commande pour la démoli...,1.000000e+11,montant_vs_pairs_aberrant,TERRES CARAIBES
1,Accord cadre à bons de commande pour la démoli...,1.000000e+11,montant_vs_pairs_aberrant,TERRES CARAIBES
2,Accord cadre à bons de commande pour la démoli...,1.000000e+11,montant_vs_pairs_aberrant,TERRES CARAIBES
3,Accord cadre à bons de commande pour la démoli...,1.000000e+11,montant_vs_pairs_aberrant,TERRES CARAIBES
4,Accord cadre à bons de commande pour la démoli...,1.000000e+11,montant_vs_pairs_aberrant,TERRES CARAIBES


## 5. Les dates

Une date de notification absente ou fantaisiste rend la ligne inutilisable pour toute analyse
temporelle, qui est le coeur du projet.

In [9]:
con.sql("""
    select
        sum(case when dateNotification is null then 1 else 0 end)             as absentes,
        min(dateNotification)                                                 as plus_ancienne,
        max(dateNotification)                                                 as plus_recente,
        sum(case when dateNotification < date '2015-01-01' then 1 else 0 end) as avant_2015,
        sum(case when dateNotification > current_date then 1 else 0 end)      as dans_le_futur
    from decp
""").df()

,absentes,plus_ancienne,plus_recente,avant_2015,dans_le_futur
0,31812.0,1-01-01,2026-09-19,1441.0,0.0


La date la plus ancienne est le **1er janvier de l'an 1**. C'est la valeur par defaut d'un champ
date mal rempli, pas une erreur de lecture. Une regle simple, une comparaison a une borne basse,
suffit a l'ecarter : premier barreau de l'echelle de complexite, aucun modele n'est necessaire.

## 6. Les offres recues, et un vrai probleme pour la suite

Le nombre d'offres recues est **l'indicateur de risque le plus utilise** dans la litterature sur la
commande publique : un marche n'ayant recu qu'une seule offre merite un regard. C'est le coeur de
la detection prevue en phase 5.

In [10]:
con.sql("""
    select
        count(*)                                             as lignes,
        sum(case when offresRecues is null then 1 else 0 end) as absentes,
        round(100.0 * sum(case when offresRecues is null then 1 else 0 end) / count(*), 1)
            as pourcentage_absentes,
        sum(case when offresRecues = 1 then 1 else 0 end)    as une_seule_offre,
        max(offresRecues)                                    as maximum
    from decp
""").df()

,lignes,absentes,pourcentage_absentes,une_seule_offre,maximum
0,3283035,1903309.0,58.0,290752.0,20300


**58 % des lignes n'ont pas cette information**, et le maximum observe (20 300 offres pour un seul
marche) montre que meme la partie renseignee contient des saisies fausses.

C'est la principale difficulte identifiee pour la suite. Trois pistes, a arbitrer en phase 5 :

1. travailler sur le sous-ensemble renseigne, en mesurant et en assumant le biais de selection ;
2. croiser avec le BOAMP, qui publie les avis d'attribution ;
3. changer d'indicateur, par exemple la concentration des attributions par acheteur, qui ne depend
   pas de ce champ.

A documenter dans un ADR avant de coder quoi que ce soit.

## 7. Parquet contre CSV : la mesure de cout

Le meme jeu est publie dans les deux formats. Il s'agit du meme contenu, pas d'un echantillon.

In [11]:
# Poids annonces par l'API de data.gouv.fr, au 2026-09-20.
parquet_mo = 247.6
csv_mo = 2595.5
print(f"Parquet : {parquet_mo:>8.1f} Mo")
print(f"CSV     : {csv_mo:>8.1f} Mo")
print(f"rapport : {csv_mo / parquet_mo:.1f} fois plus lourd en CSV")

Parquet :    247.6 Mo
CSV     :   2595.5 Mo
rapport : 10.5 fois plus lourd en CSV


## 8. Ce qu'on retient

| Constat | Consequence pour le projet |
|---|---|
| 3,28 M de lignes, 1,83 M de marches reels | la table de travail filtre l'etat actuel, l'historique est conserve a part |
| Environ 7 800 doublons residuels seulement | pas besoin d'un rapprochement flou couteux a cette etape |
| 59 588 montants negatifs ou nuls, 2 197 au-dessus du milliard | regles de validation bloquantes des l'ingestion |
| Date minimale au 1er janvier de l'an 1 | borne basse sur les dates, regle simple |
| 58 % d'offres recues absentes | decision d'architecture a prendre avant la phase 5 |
| Parquet 10,5 fois plus leger que le CSV | premiere mesure de cout du memoire |

**Suite** : ces mesures sont rejouables par `make bench-decp`, qui ecrit un JSON et un CSV dates
dans `mesures/resultats/`. Les figures du memoire seront generees a partir de ces fichiers.